# Optuna Search

# A. Setup


In [12]:
import gc, json, random, tempfile, time
from pathlib import Path
import numpy as np
import shutil
import optuna
import pandas as pd
from IPython.display import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
SEED = 42
MAX_EPOCHS_PER_PHASE = 50
EARLY_STOPPING_PATIENCE = 5
IMAGE_SIZE = (224, 224)
HIDDEN_DIM = 512
NUM_CLASSES = 4
OPTIMIZERS = ('Adam', 'AdamW')
LEARNING_RATES = (0.001, 0.0003, 0.0001)
DROPOUT_RATES = (0.4, 0.5)
FINETUNE_LEVELS = ('frozen', 'medium', 'deep')
CLASS_NAMES = ('leaf_curl', 'leaf_spot', 'yellowish', 'healthy_leaf')
CLASS_DISPLAY_NAMES = ('Leaf Curl', 'Leaf Spot', 'Yellowish', 'Healthy Leaf')
MODEL_COLORS = {'mobilenet': '#1f77b4', 'efficientnet': '#ff7f0e', 'hybrid': '#2ca02c'}
DATA_ROOT = Path('data/processed')
TRAIN_MANIFEST = Path('data/splits/train.csv')
VALIDATION_MANIFEST = Path('data/splits/validation.csv')
CHECKPOINT_DIR = Path('checkpoints')
REPORT_DIR = Path('reports')
SCREENING_PATH = REPORT_DIR / 'screening_results.csv'
HISTORY_DIR = REPORT_DIR / 'histories'
CONFUSION_DIR = REPORT_DIR / 'confusion'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# B. Data


In [13]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1), fill=0),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
VALIDATION_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def set_class_order(dataset):
    old_indices = {name: i for i, name in enumerate(dataset.classes)}
    old_to_new = {old_indices[name]: i for i, name in enumerate(CLASS_NAMES)}
    dataset.classes = list(CLASS_NAMES)
    dataset.class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
    dataset.samples = [(path, old_to_new[target]) for path, target in dataset.samples]
    dataset.imgs = dataset.samples
    dataset.targets = [target for _, target in dataset.samples]
    return dataset
train_dataset = set_class_order(datasets.ImageFolder(
    DATA_ROOT / 'train',
    transform=TRAIN_TRANSFORM
))
validation_dataset = set_class_order(datasets.ImageFolder(
    DATA_ROOT / 'validation',
    transform=VALIDATION_TRANSFORM
))
assert train_dataset.classes == validation_dataset.classes == list(CLASS_NAMES)
count_rows = []
for split, path in (
    ('train', TRAIN_MANIFEST),
    ('validation', VALIDATION_MANIFEST),
):
    counts = pd.read_csv(path)['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    count_rows.append({'Split': split, **counts.to_dict()})
display(pd.DataFrame(count_rows).set_index('Split'))

,leaf_curl,leaf_spot,yellowish,healthy_leaf
Split,,,,
train,257,281,267,281
validation,55,59,58,61


## 1. Reproducibility and device


In [14]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

## 2. Data


In [15]:
def make_evaluation_loader(dataset, batch_size):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

def build_dataloaders(batch_size, seed):
    generator = seed_everything(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )
    return (train_loader, make_evaluation_loader(validation_dataset, batch_size))

## 3. Models


In [16]:
def build_head(input_features, dropout):
    return nn.Sequential(
        nn.Linear(input_features, HIDDEN_DIM),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(HIDDEN_DIM, NUM_CLASSES)
    )

class FeatureBackbone(nn.Module):

    def __init__(self, features):
        super().__init__()
        self.features = features
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, images):
        return self.pool(self.features(images)).flatten(1)

class SingleBackboneModel(nn.Module):

    def __init__(self, features, dropout):
        super().__init__()
        self.features = features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = build_head(1280, dropout)

    def forward(self, images):
        return self.head(self.pool(self.features(images)).flatten(1))

class FusionModel(nn.Module):

    def __init__(self, mobile_features, efficient_features, dropout):
        super().__init__()
        self.mobilenet = FeatureBackbone(mobile_features)
        self.efficientnet = FeatureBackbone(efficient_features)
        self.head = build_head(2560, dropout)

    def forward(self, images):
        return self.head(torch.cat((self.mobilenet(images), self.efficientnet(images)), dim=1))

def build_base_model(model_name, dropout):
    if model_name == 'mobilenet':
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    elif model_name == 'efficientnet':
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    else:
        raise ValueError(f'Unsupported model: {model_name}')
    return SingleBackboneModel(backbone.features, dropout).to(DEVICE)

def build_hybrid_model(dropout):
    mobile = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    efficient = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    return FusionModel(mobile.features, efficient.features, dropout).to(DEVICE)

def count_parameters(model):
    return sum((parameter.numel() for parameter in model.parameters()))

## 4. Fine-tuning control


In [17]:
def apply_finetune_level(model, finetune_level):
    if not isinstance(model, FusionModel):
        for parameter in model.parameters():
            parameter.requires_grad = True
        return
    if finetune_level not in FINETUNE_LEVELS:
        raise ValueError('Hybrid scenarios require a fine-tuning level.')
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.head.parameters():
        parameter.requires_grad = True
    if finetune_level == 'frozen':
        return
    ranges = {'medium': ((14, 19), (6, 9)), 'deep': ((10, 19), (4, 9))}
    for backbone, (start, stop) in zip(
        (model.mobilenet, model.efficientnet),
        ranges[finetune_level]
    ):
        for block in backbone.features[start:stop]:
            for parameter in block.parameters():
                parameter.requires_grad = True

def set_training_mode(model, phase):
    model.train()
    if isinstance(model, FusionModel):
        if phase == 1:
            model.mobilenet.eval()
            model.efficientnet.eval()
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
    return model

def build_optimizer(model, optimizer_name, learning_rate):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if optimizer_name == 'Adam':
        return torch.optim.Adam(parameters, lr=learning_rate, weight_decay=0.0)
    if optimizer_name == 'AdamW':
        return torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=0.01)
    raise ValueError(f'Unsupported optimizer: {optimizer_name}')

## 5. Training


In [18]:
def train_one_phase(
    model,
    train_loader,
    validation_loader,
    optimizer,
    scaler,
    phase,
    show_epoch_log
):
    criterion = nn.CrossEntropyLoss()
    best_loss = float('inf')
    best_state = None
    best_epoch = 0
    wait = 0
    history = []
    for epoch in range(1, MAX_EPOCHS_PER_PHASE + 1):
        set_training_mode(model, phase)
        started = time.perf_counter()
        loss_sum = 0.0
        correct = 0
        count = 0
        for images, targets in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(targets)
            correct += (logits.argmax(1) == targets).sum().item()
            count += len(targets)
        if not count:
            raise ValueError('Training data loader is empty.')
        train_loss = loss_sum / count
        train_accuracy = correct / count
        targets_all, predictions, _, val_loss = evaluate_model(model, validation_loader)
        val_accuracy = accuracy_score(targets_all, predictions)
        seconds = time.perf_counter() - started
        history.append({
            'phase': phase,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'validation_loss': val_loss,
            'validation_accuracy': val_accuracy,
            'epoch_seconds': seconds
        })
        if show_epoch_log:
            print(
                f"phase {phase} | epoch {epoch:02d} | train loss {train_loss:.4f} "
                f"acc {train_accuracy:.4f} | val loss {val_loss:.4f} "
                f"acc {val_accuracy:.4f} | {seconds:.1f} s"
            )
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            wait = 0
            best_state = {
                key: value.detach().cpu().clone() for key,
                value in model.state_dict().items()
            }
        else:
            wait += 1
        if wait >= EARLY_STOPPING_PATIENCE:
            break
    if best_state is None:
        raise RuntimeError('Training did not produce a best state.')
    model.load_state_dict(best_state)
    if show_epoch_log:
        print(f'phase {phase} stopped at epoch {len(history)}, best epoch {best_epoch}')
    return (model, history, best_epoch)

def train_scenario(
    scenario_id,
    model_name,
    optimizer_name,
    learning_rate,
    dropout,
    batch_size,
    finetune_level=None,
    seed=42,
    show_epoch_log=False
):
    seed_everything(seed)
    train_loader, val_loader = build_dataloaders(batch_size, seed)
    model = build_hybrid_model(dropout) if model_name == 'hybrid' else build_base_model(
        model_name,
        dropout
    )
    if model_name == 'hybrid':
        apply_finetune_level(model, 'frozen')
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    started = time.perf_counter()
    optimizer = build_optimizer(model, optimizer_name, learning_rate)
    model, phase_one_history, _ = train_one_phase(
        model,
        train_loader,
        val_loader,
        optimizer,
        scaler,
        1,
        show_epoch_log
    )
    history = phase_one_history
    epochs = [len(phase_one_history)]
    if model_name == 'hybrid' and finetune_level != 'frozen':
        apply_finetune_level(model, finetune_level)
        optimizer = build_optimizer(model, optimizer_name, learning_rate)
        model, phase_two_history, _ = train_one_phase(
            model,
            train_loader,
            val_loader,
            optimizer,
            scaler,
            2,
            show_epoch_log
        )
        history = phase_one_history + phase_two_history
        epochs.append(len(phase_two_history))
    checkpoint = CHECKPOINT_DIR / f'{scenario_id}_seed{seed}.pt'
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, checkpoint)
    seconds = time.perf_counter() - started
    del train_loader, val_loader
    return (model, history, epochs, seconds)

## 6. Evaluation


In [19]:
def evaluate_model(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum = 0.0
    targets = []
    predictions = []
    probabilities = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels_device = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss_sum += criterion(logits, labels_device).item() * len(labels)
            targets.extend(labels.numpy().tolist())
            predictions.extend(logits.argmax(1).cpu().numpy().tolist())
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
    if not targets:
        raise ValueError('Evaluation data loader is empty.')
    return (
        np.asarray(targets),
        np.asarray(predictions),
        np.concatenate(probabilities),
        loss_sum / len(targets)
    )

def compute_metrics(targets, predictions, probabilities, loss):
    return {
        'Accuracy': accuracy_score(targets, predictions),
        'Precision': precision_score(targets, predictions, average='macro', zero_division=0),
        'Recall': recall_score(targets, predictions, average='macro', zero_division=0),
        'F1': f1_score(targets, predictions, average='macro', zero_division=0),
        'Loss': float(loss),
        'AUC': roc_auc_score(
            targets,
            probabilities,
            labels=np.arange(NUM_CLASSES),
            average='macro',
            multi_class='ovr'
        )
    }

def compute_per_class_metrics(true_labels, predicted_labels):
    metric_rows = []
    for class_index, class_name in enumerate(CLASS_DISPLAY_NAMES):
        metric_rows.append({
            'Class': class_name,
            'Precision': precision_score(
                true_labels,
                predicted_labels,
                labels=[class_index],
                average='macro',
                zero_division=0
            ),
            'Recall': recall_score(
                true_labels,
                predicted_labels,
                labels=[class_index],
                average='macro',
                zero_division=0
            ),
            'F1': f1_score(
                true_labels,
                predicted_labels,
                labels=[class_index],
                average='macro',
                zero_division=0
            )
        })
    return pd.DataFrame(metric_rows)

## 7. Efficiency and storage


In [20]:
def measure_efficiency(model):
    params = count_parameters(model)
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as file:
        path = Path(file.name)
    try:
        torch.save(model.state_dict(), path)
        size = path.stat().st_size / 1024 ** 2
    finally:
        path.unlink(missing_ok=True)
    model.eval()
    sample = torch.zeros(1, 3, *IMAGE_SIZE, device=DEVICE)
    timings = []
    with torch.no_grad():
        for _ in range(20):
            model(sample)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        for _ in range(100):
            start = time.perf_counter()
            model(sample)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - start) * 1000)
    return {'Params': params, 'Size (MB)': size, 'Latency (ms/img)': float(np.mean(timings))}

def format_metrics_table(metrics, efficiency, epochs, seconds):
    columns = [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'Loss',
        'AUC',
        'Params',
        'Size (MB)',
        'Latency (ms/img)',
        'Epochs',
        'Train time (s)'
    ]
    return pd.DataFrame(
        [{**metrics, **efficiency, 'Epochs': epochs, 'Train time (s)': seconds}],
        columns=columns
    ).round(4)

def create_confusion_heatmap(scenario_id, confusion):
    figure, axis = plt.subplots(figsize=(8, 6))
    image = axis.imshow(confusion, cmap="Blues")
    figure.colorbar(image, ax=axis)
    axis.set(
        xticks=np.arange(NUM_CLASSES),
        yticks=np.arange(NUM_CLASSES),
        xticklabels=CLASS_DISPLAY_NAMES,
        yticklabels=CLASS_DISPLAY_NAMES,
        xlabel="Kelas Prediksi",
        ylabel="Kelas Sebenarnya",
        title=f"Matriks Konfusi Data Validasi: {scenario_id}",
    )
    threshold = confusion.max() / 2
    for row_index in range(NUM_CLASSES):
        for column_index in range(NUM_CLASSES):
            value = confusion[row_index, column_index]
            color = "white" if value > threshold else "black"
            axis.text(
                column_index,
                row_index,
                str(value),
                ha="center",
                va="center",
                color=color,
            )
    figure.tight_layout()
    return figure

def save_result(scenario_row, history, confusion):
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    CONFUSION_DIR.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame([scenario_row])
    if SCREENING_PATH.exists():
        previous = pd.read_csv(SCREENING_PATH)
        frame = pd.concat(
            [previous[previous.scenario_id != scenario_row['scenario_id']], frame],
            ignore_index=True
        )
    frame.to_csv(SCREENING_PATH, index=False)
    pd.DataFrame(history).to_csv(HISTORY_DIR / f"{scenario_row['scenario_id']}.csv", index=False)
    figure = create_confusion_heatmap(scenario_row['scenario_id'], confusion)
    figure.savefig(CONFUSION_DIR / f"{scenario_row['scenario_id']}.png", dpi=300)
    plt.close(figure)

def show_result_tables(
    scenario_id,
    metrics=None,
    efficiency=None,
    epochs=0,
    seconds=0.0,
    confusion=None
):
    if metrics is None:
        scenario_row = pd.read_csv(SCREENING_PATH).query('scenario_id == @scenario_id').iloc[0]
        cols = [
            'Accuracy',
            'Precision',
            'Recall',
            'F1',
            'Loss',
            'AUC',
            'Params',
            'Size (MB)',
            'Latency (ms/img)',
            'Epochs',
            'Train time (s)'
        ]
        display(pd.DataFrame([scenario_row[cols].to_dict()]).round(4))
        display(Image(filename=CONFUSION_DIR / f'{scenario_id}.png'))
    else:
        display(format_metrics_table(metrics, efficiency, epochs, seconds))
        figure = create_confusion_heatmap(scenario_id, confusion)
        display(figure)
        plt.close(figure)

def show_progress():
    results = pd.read_csv(SCREENING_PATH) if SCREENING_PATH.exists() else pd.DataFrame()
    variants = (
        ('MobileNetV2', 'mobilenet', None),
        ('EfficientNetB0', 'efficientnet', None),
        ('Hybrid frozen', 'hybrid', 'frozen'),
        ('Hybrid medium', 'hybrid', 'medium'),
        ('Hybrid deep', 'hybrid', 'deep')
    )
    rows = []
    for title, name, level in variants:
        subset = results[results.status == 'ok'] if not results.empty else results
        if not subset.empty:
            subset = subset[subset.model == name]
            if level:
                subset = subset[subset.finetune_level == level]
        rows.append({'Model variant': title, 'Scenarios done': len(subset), 'Scenarios total': 12})
    display(pd.DataFrame(rows))

def discard_scenario_results(scenario_ids):
    backup_path = REPORT_DIR / f"backup_{time.strftime('%Y%m%d_%H%M')}"
    backup_path.mkdir(parents=True, exist_ok=True)
    csv_paths = list(REPORT_DIR.glob('*.csv'))
    for csv_path in csv_paths:
        csv_path.replace(backup_path / csv_path.name)
    result_files = ('screening_results.csv', 'seed_results.csv', 'test_results.csv')
    for file_name in result_files:
        backup_csv = backup_path / file_name
        if not backup_csv.exists():
            continue
        stored_results = pd.read_csv(backup_csv)
        stored_results = stored_results[~stored_results.scenario_id.isin(scenario_ids)]
        stored_results.to_csv(REPORT_DIR / file_name, index=False)
    for directory in (HISTORY_DIR, CONFUSION_DIR, CHECKPOINT_DIR):
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if any((path.name.startswith(scenario_id) for scenario_id in scenario_ids)):
                if path.is_file():
                    path.unlink()

## 8. Optuna search

In [21]:
N_TRIALS = 12
OPTUNA_STORAGE = 'sqlite:///reports/optuna.db'
OPTUNA_TABLE_COLUMNS = [
    'scenario_id', 'model', 'optimizer', 'lr', 'dropout', 'batch_size',
    'finetune_level', 'seed', 'status', 'Accuracy', 'Precision', 'Recall',
    'F1', 'Loss', 'AUC', 'Params', 'Size (MB)', 'Latency (ms/img)',
    'epochs_per_phase', 'Epochs', 'Train time (s)'
]

def create_study(model_name):
    return optuna.create_study(
        study_name=model_name,
        storage=OPTUNA_STORAGE,
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=4),
        load_if_exists=True
    )

def run_objective(trial, model_name):
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])
    learning_rate = trial.suggest_float('lr', 1e-05, 0.001, log=True)
    dropout = trial.suggest_float('dropout', 0.4, 0.5)
    finetune_level = trial.suggest_categorical(
        'finetune_level', ['frozen', 'medium', 'deep']
    ) if model_name == 'hybrid' else None
    scenario_id = f'optuna_{model_name}_{trial.number:02d}'
    trial.set_user_attr('scenario_id', scenario_id)
    trial.set_user_attr('model', model_name)
    trial.set_user_attr('finetune_level', finetune_level)
    started = time.perf_counter()
    model = None
    try:
        model, history, epochs, training_seconds = train_scenario(
            scenario_id=scenario_id, model_name=model_name,
            optimizer_name=optimizer_name, learning_rate=learning_rate,
            dropout=dropout, batch_size=32, finetune_level=finetune_level,
            seed=SEED, show_epoch_log=False
        )
        HISTORY_DIR.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(history).to_csv(
            HISTORY_DIR / f'optuna_{model_name}_{trial.number:02d}.csv', index=False
        )
        validation_loader = make_evaluation_loader(validation_dataset, 32)
        true_labels, predicted_labels, class_probabilities, loss = evaluate_model(
            model, validation_loader
        )
        metrics = compute_metrics(true_labels, predicted_labels, class_probabilities, loss)
        efficiency = measure_efficiency(model)
        trial.set_user_attr('metrics', metrics)
        trial.set_user_attr('efficiency', efficiency)
        trial.set_user_attr('epochs', sum(epochs))
        trial.set_user_attr('epochs_per_phase', epochs)
        trial.set_user_attr('train_time_seconds', training_seconds)
        trial.set_user_attr('status', 'ok')
        return metrics['F1']
    except torch.cuda.OutOfMemoryError:
        trial.set_user_attr('train_time_seconds', time.perf_counter() - started)
        trial.set_user_attr('status', 'oom')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def trial_table_row(trial, model_name=None):
    metrics = trial.user_attrs.get('metrics', {})
    efficiency = trial.user_attrs.get('efficiency', {})
    study_name = model_name or trial.user_attrs.get('model', 'unknown')
    return {
        'scenario_id': trial.user_attrs.get(
            'scenario_id', f'optuna_{study_name}_{trial.number:02d}'
        ),
        'model': trial.user_attrs.get('model', study_name),
        'optimizer': trial.params.get('optimizer'),
        'lr': trial.params.get('lr'),
        'dropout': trial.params.get('dropout'),
        'batch_size': trial.params.get('batch_size', 32),
        'finetune_level': trial.user_attrs.get(
            'finetune_level', trial.params.get('finetune_level')
        ),
        'seed': SEED,
        'status': trial.user_attrs.get(
            'status', 'ok' if trial.state == optuna.trial.TrialState.COMPLETE
            else trial.state.name.lower()
        ),
        **metrics, **efficiency,
        'epochs_per_phase': json.dumps(trial.user_attrs.get('epochs_per_phase', [])),
        'Epochs': trial.user_attrs.get('epochs'),
        'Train time (s)': trial.user_attrs.get('train_time_seconds')
    }

def export_optuna_trials(study):
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    trial_path = REPORT_DIR / 'optuna_trials.csv'
    study_frame = pd.DataFrame(
        [trial_table_row(trial, study.study_name) for trial in study.trials],
        columns=OPTUNA_TABLE_COLUMNS
    )
    if trial_path.exists():
        previous = pd.read_csv(trial_path)
        previous = previous[previous['model'] != study.study_name]
        study_frame = pd.concat([previous, study_frame], ignore_index=True)
    study_frame.reindex(columns=OPTUNA_TABLE_COLUMNS).to_csv(trial_path, index=False)

def run_optuna_study(model_name):
    study = create_study(model_name)
    remaining_trials = max(0, N_TRIALS - len(study.trials))
    if remaining_trials:
        study.optimize(
            lambda trial: run_objective(trial, model_name),
            n_trials=remaining_trials,
            catch=(torch.cuda.OutOfMemoryError,), gc_after_trial=True,
            show_progress_bar=False
        )
    export_optuna_trials(study)
    completed_trials = [
        trial for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
    ]
    ranked_trials = sorted(
        completed_trials,
        key=lambda trial: (
            -trial.value,
            trial.user_attrs.get('metrics', {}).get('Loss', float('inf'))
        )
    )
    top_rows = [trial_table_row(trial, model_name) for trial in ranked_trials[:5]]
    display(pd.DataFrame(top_rows, columns=OPTUNA_TABLE_COLUMNS).round(4))
    if ranked_trials:
        display(pd.DataFrame(
            [trial_table_row(ranked_trials[0], model_name)],
            columns=OPTUNA_TABLE_COLUMNS
        ).round(4))
    return study

def retrain_optuna_trial(study_name, trial_number):
    study = optuna.load_study(study_name=study_name, storage=OPTUNA_STORAGE)
    trial = study.trials[trial_number]
    model_name = trial.user_attrs.get('model', study_name)
    scenario_id = trial.user_attrs.get(
        'scenario_id', f'optuna_{model_name}_{trial.number:02d}'
    )
    model, history, epochs, training_seconds = train_scenario(
        scenario_id=scenario_id, model_name=model_name,
        optimizer_name=trial.params['optimizer'], learning_rate=trial.params['lr'],
        dropout=trial.params['dropout'], batch_size=trial.params.get('batch_size', 32),
        finetune_level=trial.params.get(
            'finetune_level', trial.user_attrs.get('finetune_level')
        ), seed=SEED, show_epoch_log=False
    )
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(
        HISTORY_DIR / f'optuna_{model_name}_{trial.number:02d}.csv', index=False
    )
    return model, history, epochs, training_seconds


### MobileNet Study

In [22]:
# Round 1 Optuna trials are archived and not rerun under the Round 2 protocol.
# mobilenet_study = run_optuna_study('mobilenet')

[I 2026-11-13 02:28:33,261] A new study created in RDB with name: mobilenet
[I 2026-11-13 02:30:08,746] Trial 0 finished with value: 0.8923723904042445 and parameters: {'optimizer': 'AdamW', 'lr': 0.000291063591313307, 'dropout': 0.4598658484197037}. Best is trial 0 with value: 0.8923723904042445.
[I 2026-11-13 02:36:57,574] Trial 1 finished with value: 0.9047361970843278 and parameters: {'optimizer': 'Adam', 'lr': 1.3066739238053272e-05, 'dropout': 0.48661761457749353}. Best is trial 1 with value: 0.9047361970843278.
[I 2026-11-13 02:43:44,293] Trial 2 finished with value: 0.8957002516453736 and parameters: {'optimizer': 'AdamW', 'lr': 1.0994335574766187e-05, 'dropout': 0.49699098521619944}. Best is trial 1 with value: 0.9047361970843278.
[I 2026-11-13 02:48:35,870] Trial 3 finished with value: 0.9049725401971812 and parameters: {'optimizer': 'Adam', 'lr': 2.3102018878452926e-05, 'dropout': 0.4183404509853434}. Best is trial 3 with value: 0.9049725401971812.
[I 2026-11-13 02:53:29,322

,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_mobilenet_09,mobilenet,Adam,0.0001,0.4713,32,None,42,ok,0.9227,...,0.9219,0.9221,0.2420,0.9888,2881796,11.2284,11.7377,[15],15,130.7600
1,optuna_mobilenet_07,mobilenet,Adam,0.0001,0.4917,32,None,42,ok,0.9185,...,0.9183,0.9181,0.2350,0.9915,2881796,11.2284,7.0783,[18],18,145.4485
2,optuna_mobilenet_08,mobilenet,Adam,0.0001,0.4986,32,None,42,ok,0.9142,...,0.9138,0.9136,0.2621,0.9885,2881796,11.2284,8.1157,[24],24,205.2869
3,optuna_mobilenet_11,mobilenet,Adam,0.0001,0.4687,32,None,42,ok,0.9099,...,0.9095,0.9100,0.2557,0.9876,2881796,11.2284,7.0855,[14],14,122.0461
4,optuna_mobilenet_03,mobilenet,Adam,0.0000,0.4183,32,None,42,ok,0.9056,...,0.9051,0.9050,0.2478,0.9888,2881796,11.2284,7.8499,[36],36,288.7786


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_mobilenet_09,mobilenet,Adam,0.0001,0.4713,32,None,42,ok,0.9227,...,0.9219,0.9221,0.242,0.9888,2881796,11.2284,11.7377,[15],15,130.76


### EfficientNet Study

In [23]:
# Round 1 Optuna trials are archived and not rerun under the Round 2 protocol.
# efficientnet_study = run_optuna_study('efficientnet')

[I 2026-11-13 03:11:15,309] A new study created in RDB with name: efficientnet
[I 2026-11-13 03:13:08,487] Trial 0 finished with value: 0.9321344771429981 and parameters: {'optimizer': 'AdamW', 'lr': 0.000291063591313307, 'dropout': 0.4598658484197037}. Best is trial 0 with value: 0.9321344771429981.
[I 2026-11-13 03:21:35,687] Trial 1 finished with value: 0.9133387868421705 and parameters: {'optimizer': 'Adam', 'lr': 1.3066739238053272e-05, 'dropout': 0.48661761457749353}. Best is trial 0 with value: 0.9321344771429981.
[I 2026-11-13 03:29:17,990] Trial 2 finished with value: 0.9046948437161443 and parameters: {'optimizer': 'AdamW', 'lr': 1.0994335574766187e-05, 'dropout': 0.49699098521619944}. Best is trial 0 with value: 0.9321344771429981.
[I 2026-11-13 03:34:45,781] Trial 3 finished with value: 0.9352754393738001 and parameters: {'optimizer': 'Adam', 'lr': 2.3102018878452926e-05, 'dropout': 0.4183404509853434}. Best is trial 3 with value: 0.9352754393738001.
[I 2026-11-13 03:40:17,

,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_efficientnet_06,efficientnet,Adam,0.0004,0.4431,32,None,42,ok,0.9356,...,0.9364,0.9358,0.1963,0.9937,4665472,18.084,10.7786,[14],14,125.0334
1,optuna_efficientnet_03,efficientnet,Adam,0.0000,0.4183,32,None,42,ok,0.9356,...,0.9352,0.9353,0.2243,0.9914,4665472,18.084,11.2187,[36],36,324.5739
2,optuna_efficientnet_00,efficientnet,AdamW,0.0003,0.4599,32,None,42,ok,0.9313,...,0.9310,0.9321,0.1809,0.9958,4665472,18.084,11.2575,[9],9,109.7686
3,optuna_efficientnet_10,efficientnet,AdamW,0.0000,0.4382,32,None,42,ok,0.9313,...,0.9310,0.9310,0.2287,0.9919,4665472,18.084,13.3607,[28],28,274.1974
4,optuna_efficientnet_09,efficientnet,AdamW,0.0001,0.4060,32,None,42,ok,0.9313,...,0.9306,0.9309,0.1960,0.9947,4665472,18.084,14.7612,[12],12,116.9704


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_efficientnet_06,efficientnet,Adam,0.0004,0.4431,32,None,42,ok,0.9356,...,0.9364,0.9358,0.1963,0.9937,4665472,18.084,10.7786,[14],14,125.0334


### Hybrid Study

In [24]:
# Round 1 Optuna trials are archived and not rerun under the Round 2 protocol.
# hybrid_study = run_optuna_study('hybrid')

[I 2026-11-13 03:57:13,240] A new study created in RDB with name: hybrid
[I 2026-11-13 04:02:07,022] Trial 0 finished with value: 0.90476957612919 and parameters: {'optimizer': 'AdamW', 'lr': 0.000291063591313307, 'dropout': 0.4598658484197037, 'finetune_level': 'frozen'}. Best is trial 0 with value: 0.90476957612919.
[I 2026-11-13 04:07:34,683] Trial 1 finished with value: 0.9095697308730623 and parameters: {'optimizer': 'Adam', 'lr': 0.0002607024758370766, 'dropout': 0.40205844942958024, 'finetune_level': 'frozen'}. Best is trial 1 with value: 0.9095697308730623.
[I 2026-11-13 04:20:51,430] Trial 2 finished with value: 0.9351280514730875 and parameters: {'optimizer': 'AdamW', 'lr': 4.059611610484306e-05, 'dropout': 0.4524756431632238, 'finetune_level': 'deep'}. Best is trial 2 with value: 0.9351280514730875.
[I 2026-11-13 04:27:13,087] Trial 3 finished with value: 0.8776876129852201 and parameters: {'optimizer': 'AdamW', 'lr': 5.4041038546473305e-05, 'dropout': 0.4456069984217036, 'f

,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_hybrid_06,hybrid,AdamW,0.0008,0.4042,32,deep,42,ok,0.9571,...,0.9570,0.9568,0.1713,0.9945,7544704,29.3144,17.9905,"[21, 14]",35,269.5224
1,optuna_hybrid_05,hybrid,AdamW,0.0008,0.4195,32,deep,42,ok,0.9485,...,0.9484,0.9485,0.1681,0.9951,7544704,29.3144,18.1242,"[26, 17]",43,328.6915
2,optuna_hybrid_10,hybrid,Adam,0.0002,0.4647,32,deep,42,ok,0.9442,...,0.9442,0.9438,0.2039,0.9929,7544704,29.3144,18.5060,"[50, 24]",74,559.2424
3,optuna_hybrid_08,hybrid,AdamW,0.0001,0.4838,32,deep,42,ok,0.9356,...,0.9357,0.9353,0.2112,0.9923,7544704,29.3144,17.3234,"[50, 24]",74,559.3705
4,optuna_hybrid_02,hybrid,AdamW,0.0000,0.4525,32,deep,42,ok,0.9356,...,0.9353,0.9351,0.2404,0.9881,7544704,29.3144,19.3399,"[50, 42]",92,791.9493


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_hybrid_06,hybrid,AdamW,0.0008,0.4042,32,deep,42,ok,0.9571,...,0.957,0.9568,0.1713,0.9945,7544704,29.3144,17.9905,"[21, 14]",35,269.5224


## E. Grid and Optuna Comparison

In [25]:
def compare_grid_and_optuna():
    grid_results = None
    if SCREENING_PATH.exists():
        grid_results = pd.read_csv(SCREENING_PATH)
    else:
        print(f'Grid screening results not found at {SCREENING_PATH.resolve()}')
    comparison_rows = []
    model_names = {
        'mobilenet': 'MobileNetV2', 'efficientnet': 'EfficientNetB0',
        'hybrid': 'Hybrid'
    }
    for model_name, display_name in model_names.items():
        if grid_results is None:
            model_runs = pd.DataFrame()
            best_grid = None
            grid_seconds = 0.0
        else:
            model_runs = grid_results[grid_results.model == model_name]
            model_grid = model_runs[model_runs.status == 'ok']
            grid_seconds = model_runs['Train time (s)'].sum()
            best_grid = None if model_grid.empty else model_grid.sort_values(
                ['F1', 'Loss'], ascending=[False, True], kind='stable'
            ).iloc[0]
        study = optuna.load_study(study_name=model_name, storage=OPTUNA_STORAGE)
        completed_trials = [
            trial for trial in study.trials
            if trial.state == optuna.trial.TrialState.COMPLETE
        ]
        best_trial = min(
            completed_trials,
            key=lambda trial: (
                -trial.value,
                trial.user_attrs.get('metrics', {}).get('Loss', float('inf'))
            ), default=None
        )
        optuna_metrics = best_trial.user_attrs.get('metrics', {}) if best_trial else {}
        optuna_seconds = sum(
            trial.user_attrs.get('train_time_seconds', 0.0)
            for trial in study.trials
        )
        comparison_rows.append({
            'Model type': display_name,
            'Grid scenario': best_grid.scenario_id if best_grid is not None else None,
            'Grid F1': best_grid.F1 if best_grid is not None else np.nan,
            'Grid Loss': best_grid.Loss if best_grid is not None else np.nan,
            'Grid AUC': best_grid.AUC if best_grid is not None else np.nan,
            'Grid optimizer': best_grid.optimizer if best_grid is not None else None,
            'Grid lr': best_grid.lr if best_grid is not None else np.nan,
            'Grid dropout': best_grid.dropout if best_grid is not None else np.nan,
            'Grid fine-tuning level': (
                best_grid.finetune_level
                if best_grid is not None and pd.notna(best_grid.finetune_level) else '-'
            ),
            'Grid runs': len(model_runs),
            'Grid total training time (s)': grid_seconds,
            'Optuna trial': best_trial.number if best_trial is not None else None,
            'Optuna F1': best_trial.value if best_trial is not None else np.nan,
            'Optuna Loss': optuna_metrics.get('Loss', np.nan),
            'Optuna AUC': optuna_metrics.get('AUC', np.nan),
            'Optuna optimizer': (
                best_trial.params.get('optimizer') if best_trial is not None else None
            ),
            'Optuna lr': best_trial.params.get('lr') if best_trial is not None else np.nan,
            'Optuna dropout': (
                best_trial.params.get('dropout') if best_trial is not None else np.nan
            ),
            'Optuna fine-tuning level': (
                best_trial.user_attrs.get('finetune_level', '-')
                if best_trial is not None else '-'
            ),
            'Optuna runs': len(study.trials),
            'Optuna total training time (s)': optuna_seconds
        })
    comparison = pd.DataFrame(comparison_rows).round(4)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(REPORT_DIR / 'optuna_vs_grid.csv', index=False)
    display(comparison)
    return comparison

compare_grid_and_optuna()


,Model type,Grid scenario,Grid F1,Grid Loss,Grid AUC,Grid optimizer,Grid lr,Grid dropout,Grid fine-tuning level,Grid runs,...,Optuna trial,Optuna F1,Optuna Loss,Optuna AUC,Optuna optimizer,Optuna lr,Optuna dropout,Optuna fine-tuning level,Optuna runs,Optuna total training time (s)
0,MobileNetV2,mobilenet_09,0.9269,0.2419,0.9887,AdamW,0.0001,0.4,-,12,...,9,0.9221,0.2420,0.9888,Adam,0.0001,0.4713,NaN,12,2527.0738
1,EfficientNetB0,efficientnet_10,0.9400,0.2156,0.9935,AdamW,0.0001,0.5,-,12,...,6,0.9358,0.1963,0.9937,Adam,0.0004,0.4431,NaN,12,2715.9421
2,Hybrid,hybrid_01,0.9182,0.2529,0.9879,Adam,0.0010,0.4,frozen,12,...,6,0.9568,0.1713,0.9945,AdamW,0.0008,0.4042,deep,12,6071.2860


,Model type,Grid scenario,Grid F1,Grid Loss,Grid AUC,Grid optimizer,Grid lr,Grid dropout,Grid fine-tuning level,Grid runs,...,Optuna trial,Optuna F1,Optuna Loss,Optuna AUC,Optuna optimizer,Optuna lr,Optuna dropout,Optuna fine-tuning level,Optuna runs,Optuna total training time (s)
0,MobileNetV2,mobilenet_09,0.9269,0.2419,0.9887,AdamW,0.0001,0.4,-,12,...,9,0.9221,0.2420,0.9888,Adam,0.0001,0.4713,NaN,12,2527.0738
1,EfficientNetB0,efficientnet_10,0.9400,0.2156,0.9935,AdamW,0.0001,0.5,-,12,...,6,0.9358,0.1963,0.9937,Adam,0.0004,0.4431,NaN,12,2715.9421
2,Hybrid,hybrid_01,0.9182,0.2529,0.9879,Adam,0.0010,0.4,frozen,12,...,6,0.9568,0.1713,0.9945,AdamW,0.0008,0.4042,deep,12,6071.2860


## 9. Optuna figures

In [26]:
def plot_optuna_studies(study_names=('mobilenet', 'efficientnet', 'hybrid')):
    import warnings
    from optuna.exceptions import ExperimentalWarning
    from optuna.visualization import matplotlib as optuna_matplotlib
    figure_directory = REPORT_DIR / 'figures'
    figure_directory.mkdir(parents=True, exist_ok=True)
    parameter_labels = {
        'dropout': 'Dropout',
        'lr': 'Laju pembelajaran (skala log)',
        'optimizer': 'Pengoptimal',
        'finetune_level': 'Tingkat fine-tuning'
    }
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ExperimentalWarning)
        for study_name in study_names:
            study = optuna.load_study(study_name=study_name, storage=OPTUNA_STORAGE)
            study_color = MODEL_COLORS[study_name]
            history_axis = optuna_matplotlib.plot_optimization_history(study)
            history_axis.set_title(f'Riwayat Optimasi: {study_name}')
            history_axis.set_xlabel('Percobaan')
            history_axis.set_ylabel('Macro F1 Validasi')
            for line in history_axis.lines:
                if line.get_label() == 'Objective Value':
                    line.set_label('Nilai Objektif')
                elif line.get_label() == 'Best Value':
                    line.set_label('Nilai Terbaik')
                line.set_color(study_color)
            history_axis.legend(loc='lower right')
            history_figure = history_axis.figure
            history_figure.set_layout_engine('constrained')
            history_figure.savefig(
                figure_directory / f'optuna_optimization_history_{study_name}.png', dpi=300
            )
            plt.close(history_figure)

            slice_axes = np.asarray(optuna_matplotlib.plot_slice(study), dtype=object).reshape(-1)
            slice_figure = slice_axes[0].figure
            for axis in slice_axes:
                parameter_name = axis.get_xlabel().strip()
                label = parameter_labels.get(parameter_name, parameter_name)
                axis.set_xlabel(label)
                axis.set_ylabel('Macro F1 Validasi')
                axis.set_title('')
                axis.set_xticks(axis.get_xticks())
                axis.set_xticklabels([
                    tick.get_text().replace("'", '') for tick in axis.get_xticklabels()
                ])
                for collection in axis.collections:
                    collection.set_color(study_color)
            for colorbar_axis in slice_figure.axes[len(slice_axes):]:
                colorbar_axis.set_ylabel('Percobaan')
            slice_figure.suptitle(f'Plot Irisan: {study_name}')
            slice_figure.set_layout_engine('constrained')
            slice_figure.savefig(
                figure_directory / f'optuna_slice_{study_name}.png', dpi=300
            )
            plt.close(slice_figure)

            try:
                importance_axis = optuna_matplotlib.plot_param_importances(study)
            except (ValueError, RuntimeError, ZeroDivisionError):
                continue
            importance_axis.set_title('', loc='left')
            importance_axis.set_title(f'Kepentingan Parameter: {study_name}', loc='left')
            importance_axis.set_xlabel('Kepentingan')
            importance_axis.set_ylabel('Parameter')
            importance_axis.set_yticklabels([
                parameter_labels.get(label.get_text(), label.get_text())
                for label in importance_axis.get_yticklabels()
            ])
            for line in importance_axis.lines:
                if line.get_label() == 'Objective Value':
                    line.set_label('Macro F1 Validasi')
            importance_figure = importance_axis.figure
            importance_figure.set_layout_engine('constrained')
            importance_figure.savefig(
                figure_directory / f'optuna_parameter_importance_{study_name}.png', dpi=300
            )
            plt.close(importance_figure)

plot_optuna_studies()
